In [ ]:
!pip install transformers peft accelerate bitsandbytes pandas tqdm

In [ ]:
!pip install rouge-score bert-score

In [ ]:

import torch
import math
import json
import random
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

## JSONL path 경로 추가 할것!!!

In [ ]:
BASE_MODEL = 'google/gemma-3-12b-it'
ADAPTER_MODEL = "YHPark0208/SKN24_3rd_2Team"

# ************** 추가!!
JSONL_PATH    = ""   # ← 추*****************************가


### 질문 넣어주기

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
# use_stemmer=False로 설정하여 어간 추출을 비활성화


# rouge-l 계산 함수

def calc_rouge_l(reference:str, hypothesis:str) -> float:
    # reference : 정답 텍스트
    # hypothesis : 모델이 생성한 텍스트

    
    score = rouge.score(reference, hypothesis)
    return round(score['rougeL'].fmeasure,4)


# bert score 계산 함수
def calc_bert_score(references:list, hypothesis:list) -> list:
    _, _, F1 = bert_score_fn(
        hypothesis,
        references,
        lang="ko",
        verbose=False
    )
    return [round(f.item(),4) for f in F1]



# Perplexity 계산 함수
def calc_perplexity(model, text:str)-> float:
    inputs = tokenizer(
        text,
        return_tensors = "pt",
        truncation = True,
        max_length = 512
    ).to(model.device)

    input_ids = inputs["input_ids"]

    if input_ids.shape[1] < 2:
        return float("inf")

    with torch.no_grad():
        outputs = model(
            input_ids = input_ids,
            labels = input_ids
        )
    loss = outputs.loss.item()
    return round(math.exp(loss),4)



def get_answer(model,question : str) -> str:
    messages = [{
        "role": "user",
        "content": f"다음 F1 규정 질문에 한국어로 답변하세요.\n\n질문: {question}"
    }]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt = True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512,
            do_sample = False,
            pad_token_id = tokenizer.eos_token_id 
            # 생성된 토큰이 패딩 토큰이 되도록 설정 
        )

    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    ).strip()




In [ ]:

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line.strip())
            if "messages" in item:
                msgs = item["messages"]
                q = next((m["content"] for m in msgs if m["role"] == "user"), None)
                a = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
            elif "instruction" in item:
                q = item["instruction"]
                a = item.get("output", "")
            else:
                continue
            if q and a:
                data.append({"question": q, "reference": a})
    return data

all_data = load_jsonl(JSONL_PATH)


sampled  = all_data # 전체 데이터 사용
print(f"✅ {len(sampled)}개 샘플 로드 완료")
print(f"예시 질문  : {sampled[0]['question'][:50]}...")
print(f"예시 정답  : {sampled[0]['reference'][:50]}...")

In [ ]:
print('베이스 모델 로딩')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_8bit = True,
    device_map = "auto",
    torch_dtype = torch.float16
)

base_model.eval() # 베이스 모델 로드


base_results = []

for item in tqdm(sampled, desc="베이스 모델 평가"):
    q   = item["question"]
    ref = item["reference"]
    ans = get_answer(base_model, q)
    ppl = calc_perplexity(base_model, ans)
    rl  = calc_rouge_l(ref, ans)
    base_results.append({
        "question":  q,
        "reference": ref,
        "answer":    ans,
        "ppl":       ppl,
        "rouge_l":   rl
    })
del base_model
torch.cuda.empty_cache() # 베이스 모델 해제
print('베이스 모델 메모리 해제')

In [ ]:
print('파인튜닝 모델 로딩')
ft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_4bit = True,
    device_map = "auto",
    torch_dtype = torch.float16
)
ft_model = PeftModel.from_pretrained(
    ft_model, ADAPTER_MODEL)
ft_model.eval()
print('파인튜닝 모델 로드 완료')

ft_results = []
for item in tqdm(sampled, desc="파인튜닝 모델 평가"):
    q   = item["question"]
    ref = item["reference"]
    ans = get_answer(ft_model, q)
    ppl = calc_perplexity(ft_model, ans)
    rl  = calc_rouge_l(ref, ans)
    ft_results.append({
        "question":  q,
        "reference": ref,
        "answer":    ans,
        "ppl":       ppl,
        "rouge_l":   rl
    })
del ft_model
torch.cuda.empty_cache() # 파인튜닝 모델 해제
print('파인튜닝 모델 메모리 해제')

In [ ]:
# bert score 계산
print('Bert Score 계산')

# 베이스 모델 Bert Score 계산
base_bert = calc_bert_score(
    references = [r['reference'] for r in base_results],
    hypotheses = [r['answer'] for r in base_results]
)

# 파인튜닝 모델 Bert Score 계산
ft_bert = calc_bert_score(
    references = [r['reference'] for r in ft_results],
    hypotheses = [r['answer'] for r in ft_results]
)

for i in range(len(base_results)):
    base_results[i]['bert_score'] = base_bert[i]
    ft_results[i]['bert_score'] = ft_bert[i]

print('BERTScore 계산 완료')

### 결과 

In [ ]:
# 셀 8 — 결과 합치기 & 저장
rows = []
for b, f in zip(base_results, ft_results):
    rows.append({
        "question"        : b["question"],
        "reference"       : b["reference"],
        "base_answer"     : b["answer"],
        "ft_answer"       : f["answer"],
        "base_ppl"        : b["ppl"],
        "ft_ppl"          : f["ppl"],
        "base_rouge_l"    : b["rouge_l"],
        "ft_rouge_l"      : f["rouge_l"],
        "base_bert_score" : b["bert_score"],
        "ft_bert_score"   : f["bert_score"],
    })

df = pd.DataFrame(rows)
df.to_csv("eval_results.csv", index=False, encoding="utf-8-sig")
print("✅ eval_results.csv 저장 완료")

In [ ]:
# 셀 9 — 최종 결과 출력
print("📊 베이스 vs 파인튜닝 비교 결과")
print("=" * 50)
print(f"{'지표':<15} {'베이스':>10} {'파인튜닝':>10} {'개선':>10}")
print("-" * 50)

base_ppl  = df['base_ppl'].mean()
ft_ppl    = df['ft_ppl'].mean()
base_rl   = df['base_rouge_l'].mean()
ft_rl     = df['ft_rouge_l'].mean()
base_bert = df['base_bert_score'].mean()
ft_bert   = df['ft_bert_score'].mean()

print(f"{'Perplexity':<15} {base_ppl:>10.2f} {ft_ppl:>10.2f} {base_ppl - ft_ppl:>+10.2f}")
print(f"{'ROUGE-L':<15} {base_rl:>10.4f} {ft_rl:>10.4f} {ft_rl - base_rl:>+10.4f}")
print(f"{'BERTScore':<15} {base_bert:>10.4f} {ft_bert:>10.4f} {ft_bert - base_bert:>+10.4f}")
print()
print("Perplexity 개선  →  낮아질수록 좋음")
print("ROUGE-L 개선     →  높아질수록 좋음")
print("BERTScore 개선   →  높아질수록 좋음")